# Рекуррентные нейронные сети

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Рекуррентные нейронные сети».

## Научная цель

Рекуррентная нейронная сеть предназначена для данных, в которых важен порядок: временных рядов, сигналов, последовательностей измерений. Сеть обрабатывает элементы по одному и хранит внутреннее состояние — память о предыстории. Это позволяет учёному моделировать процессы, разворачивающиеся во времени, и работать с записями переменной длины.

В этом блокноте мы обучим управляемую рекуррентную сеть (LSTM) прогнозировать временной ряд на шаг вперёд. Особое внимание уделим корректному разбиению данных по времени — без заглядывания в будущее. Блокнот исполняется на CPU за десятки секунд.

## Интуиция за методом

Представьте, что вы читаете дневник погоды: сегодняшнее давление лучше понять, если знать вчерашнее и позавчерашнее. Обычная нейронная сеть смотрит только на одну запись изолированно. Рекуррентная сеть читает дневник по порядку, сохраняя «конспект» всего прочитанного и обновляя его с каждым новым значением. Этот конспект — **скрытое состояние** — передаётся от шага к шагу.

Простая рекуррентная сеть плохо «помнит» события, случившиеся давно: сигнал затухает при распространении через множество шагов. Поэтому на практике используют **LSTM** (Long Short-Term Memory — долгая кратковременная память): у неё есть специальные «ворота», регулирующие, что запомнить надолго, что забыть и что передать дальше. Это похоже на того же читателя дневника, который выделяет маркером важное и зачёркивает устаревшее.

**Ключевые термины, которые встретятся в блокноте:**

- **Скрытое состояние (hidden state)** — вектор числел, представляющий «память» сети о прошлых шагах.
- **LSTM** — разновидность рекуррентной сети с управляемыми воротами, устойчивая к затуханию сигнала на длинных последовательностях.
- **Окно (window)** — отрезок ряда фиксированной длины, по которому сеть учится предсказывать следующее значение.
- **Эпоха** — один полный проход обучающего набора; нужно несколько десятков, чтобы ошибка перестала снижаться.
- **Обрезка градиентов (gradient clipping)** — ограничение длины вектора градиента, предотвращающее «взрыв» обновлений весов.
- **Утечка данных из будущего** — ситуация, когда модель косвенно получает информацию о будущих значениях при обучении; ведёт к завышенной оценке качества. Устраняется строгим разбиением по времени.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 scikit-learn==1.5.1 numpy==1.26.4 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем синтетический временной ряд с сезонностью, плавным трендом и шумом — типичная структура научного сигнала или ряда наблюдений. Разобьём его по времени: ранняя часть — для обучения, поздняя — для проверки.

**Почему нельзя перемешивать случайно?** Случайное перемешивание привело бы к тому, что сеть во время обучения видит значения «из будущего» относительно тестового набора — это и есть утечка данных. Качество на тесте окажется необоснованно высоким, а модель в реальных условиях будет плохо работать.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
T = 600
t = np.arange(T)
# Сезонность двух периодов + плавный тренд + шум.
series = (np.sin(2 * np.pi * t / 50)
          + 0.5 * np.sin(2 * np.pi * t / 17)
          + 0.002 * t
          + 0.15 * rng.standard_normal(T)).astype('float32')

split = 460                       # граница обучение / проверка
print(f'Длина ряда: {T}, обучение: 0-{split}, проверка: {split}-{T}')

### Как выглядит ряд

Прежде чем подавать данные в модель, всегда полезно посмотреть на них. Ячейка ниже показывает весь ряд и границу обучение/проверка.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(split), series[:split], color=VIZ["series"][0],
        lw=1.5, label="Обучающая часть")
ax.plot(range(split, T), series[split:], color=VIZ["series"][2],
        lw=1.5, label="Тестовая часть")
ax.axvline(split, color=VIZ["edge"], lw=1.2, linestyle="--",
           label=f"Граница разбиения (t={split})")
ax.set_title("Синтетический временной ряд с сезонностью и трендом")
ax.set_xlabel("Время (условные единицы)")
ax.set_ylabel("Значение ряда")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

### Формирование обучающих окон

Рекуррентная сеть учится по **скользящим окнам**: по отрезку из `WINDOW` последовательных значений предсказать следующее (один шаг вперёд). Нормировку (вычитание среднего, деление на стандартное отклонение) считаем **только по обучающей части** — иначе статистика будущего «протечёт» в модель через нормировочные коэффициенты.

In [ ]:
WINDOW = 40

# Нормировка по обучающей части.
mean = series[:split].mean()
std = series[:split].std()
norm = (series - mean) / std


def make_windows(seq, start, end):
    """Окна (вход, следующий шаг) в пределах [start, end)."""
    xs, ys = [], []
    for i in range(start, end - WINDOW):
        xs.append(seq[i:i + WINDOW])
        ys.append(seq[i + WINDOW])
    return (np.array(xs, dtype='float32'),
            np.array(ys, dtype='float32'))


X_train, y_train = make_windows(norm, 0, split)
X_test, y_test = make_windows(norm, split - WINDOW, T)
print(f'Обучающих окон: {len(X_train)}, тестовых: {len(X_test)}')

## 4. Применение метода

### Шаг 1. Архитектура LSTM-сети

Соберём сеть с рекуррентным слоем **LSTM**. Почему LSTM, а не простая рекуррентная сеть?

В простой рекуррентной сети сигнал при распространении через много шагов либо затухает до нуля (сеть «забывает» давние события), либо взрывается (обучение расходится). LSTM решает эту проблему с помощью **ворот** — обучаемых механизмов, которые решают:
- что из прошлого забыть (ворота забывания),
- что из нового входа запомнить (входные ворота),
- что передать наружу на текущем шаге (выходные ворота).

Выход последнего временного шага передаётся на линейный слой-предсказатель, который возвращает одно число — следующее значение ряда.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


class LSTMForecaster(nn.Module):
    """Прогноз временного ряда на один шаг вперёд."""

    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden,
                            num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: (батч, длина окна) -> (батч, длина окна, 1)
        out, _ = self.lstm(x.unsqueeze(-1))
        return self.head(out[:, -1, :]).squeeze(-1)


model = LSTMForecaster()
print(model)

### Шаг 2. Цикл обучения

Ячейка ниже обучает модель. Обратите внимание на строку `nn.utils.clip_grad_norm_`: это **обрезка градиентов** — стандартный приём для рекуррентных сетей. Без неё градиенты могут резко «взорваться» и вывести параметры сети в бесконечность. Строка ограничивает их норму значением 1.0.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

ds = TensorDataset(torch.from_numpy(X_train),
                   torch.from_numpy(y_train))
loader = DataLoader(ds, batch_size=32, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=5e-3)
crit = nn.MSELoss()

Xte = torch.from_numpy(X_test)
yte = torch.from_numpy(y_test)

history = {'train': [], 'test': []}
for epoch in range(1, 41):
    model.train()
    ep = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        # Обрезка градиентов — стандартный приём для рекуррентных сетей.
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        ep += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        te = crit(model(Xte), yte).item()
    history['train'].append(ep / len(X_train))
    history['test'].append(te)
    if epoch % 10 == 0:
        print(f'Эпоха {epoch:2d}: потеря обучение {history["train"][-1]:.4f}, тест {te:.4f}')

### Шаг 3. Визуализация: кривые обучения и прогноз против факта

Слева — кривые потерь: как снижается ошибка по эпохам. Справа — прогноз сети на тестовом участке поверх истинного ряда: насколько точно сеть следует реальной кривой.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

model.eval()
with torch.no_grad():
    pred_norm = model(Xte).numpy()
# Возврат к исходной шкале ряда.
pred = pred_norm * std + mean
true = y_test * std + mean
mae = mean_absolute_error(true, pred)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
epochs = range(1, len(history['train']) + 1)
axes[0].plot(epochs, history['train'], color=VIZ['series'][0],
             label='Обучение')
axes[0].plot(epochs, history['test'], color=VIZ['series'][2],
             label='Проверка')
axes[0].set_title('Кривые обучения')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Среднеквадратичная потеря')
axes[0].legend(loc='upper right')

axis_t = np.arange(split, T)
axes[1].plot(axis_t, true, color=VIZ['series'][3],
             label='Истинный ряд')
axes[1].plot(axis_t, pred, color=VIZ['series'][1],
             label='Прогноз сети')
axes[1].set_title(f'Прогноз на один шаг (MAE = {mae:.3f})')
axes[1].set_xlabel('Время')
axes[1].set_ylabel('Значение ряда')
axes[1].legend(loc='upper right')

fig.tight_layout()
plt.show()

### Как читать эти графики

**Кривые обучения (левый график):** обе линии должны снижаться и сходиться. Если потеря на тесте остаётся высокой при низкой потере на обучении — переобучение. Если обе высокие — модель недообучена (попробуйте больше эпох или шире сеть).

**Прогноз против факта (правый график):** чем ближе синяя линия прогноза к серой линии истинного ряда, тем лучше. Обратите внимание:
- Небольшое **запаздывание** (линия прогноза чуть сдвинута вправо) — частый эффект: модель «эхом» повторяет прошлые значения.
- **Систематический сдвиг** по вертикали говорит о том, что сеть не уловила тренд: попробуйте сначала убрать тренд из ряда вручную.
- **MAE** (средняя абсолютная ошибка) выражена в тех же единицах, что и ряд: легко оценить, насколько предсказание практически полезно.

## 5. Интерпретация результата

- **Кривые обучения**: обе линии снижаются — сеть учится. Рост потери на проверке означал бы переобучение.
- **Прогноз против факта**: линия прогноза должна повторять форму истинного ряда. Систематический сдвиг или запаздывание указывают на недостаточную ёмкость сети или слишком короткое окно.
- **MAE** выражена в единицах самого ряда и легко интерпретируется.
- Показан прогноз на **один шаг вперёд**. Многошаговый прогноз (подстановка собственных предсказаний на вход) накапливает ошибку — его горизонт надо оценивать отдельно.

Ключевое правило: данные временного ряда разбивают **по времени**, а не случайно. Для очень длинных зависимостей рекуррентные сети уступают трансформерам и моделям пространства состояний.

## 6. Попробуйте на своих данных

1. Подготовьте одномерный временной ряд — массив значений в порядке наблюдения.
2. Снимите комментарии в ячейке ниже и загрузите свой ряд.
3. Обязательно задайте границу `split` по времени и считайте нормировку только по обучающей части.
4. Подберите длину окна `WINDOW` под характерный масштаб зависимостей в ваших данных, затем выполните разделы 3 и 4.

In [ ]:
# --- Шаблон загрузки своего временного ряда ---
# import pandas as pd
#
# df = pd.read_csv('my_series.csv')
# series = df['значение'].to_numpy(dtype='float32')
# T = len(series)
# split = int(0.8 * T)            # 80 % на обучение по времени
#
# print(f'Длина ряда: {T}')
# Далее повторите шаги нормировки и формирования окон из раздела 3.

## 7. Поэкспериментируйте сами

Ниже перечислены конкретные изменения, которые стоит попробовать — каждое занимает 1–2 минуты перезапуска.

**Эксперимент 1. Длина окна.**
В ячейке нормировки измените `WINDOW = 40` на `WINDOW = 10` (короткая память) или `WINDOW = 80` (длинная память). Как изменились MAE и форма прогноза?

**Эксперимент 2. Простая RNN вместо LSTM.**
В классе `LSTMForecaster` замените `nn.LSTM(...)` на `nn.RNN(input_size=1, hidden_size=hidden, batch_first=True)` и обновите forward (у RNN выход `out, _` аналогичен). Сравните качество и кривые обучения с LSTM.

**Эксперимент 3. Размер скрытого слоя.**
Создайте модель с `LSTMForecaster(hidden=8)` вместо 32. Что происходит с качеством?

**Эксперимент 4. Случайное перемешивание (нарочно неправильно).**
В `DataLoader` уберите `shuffle=True` и перепостройте обучение, случайно перемешав индексы ряда: `X_bad, y_bad = make_windows(norm, 0, T)` с последующим `np.random.shuffle(idx)` и разбиением по idx. Насколько завышается оценка MAE на тесте по сравнению с правильным разбиением? Это наглядная демонстрация утечки данных.

## Краткие выводы

- Рекуррентные сети обрабатывают последовательности шаг за шагом, сохраняя скрытое состояние — «конспект» предыстории.
- Для длинных зависимостей используйте LSTM или GRU: они устойчивы к затуханию градиентов благодаря воротам.
- **Никогда не перемешивайте временной ряд случайно** при разбиении на обучение и тест — это приводит к утечке данных и завышенной оценке качества.
- Нормировку считайте только по обучающей части ряда.
- Прогноз на один шаг обычно точнее, чем многошаговый: при итеративной подстановке ошибки накапливаются.
- Для очень длинных зависимостей и при доступном GPU рассмотрите трансформеры или модели пространства состояний (Mamba).

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Кривая потерь на обучающей выборке снижается до эпохи 25, а кривая на тестовой выборке после эпохи 15 начинает расти. Что происходит и какие два параметра вы измените в первую очередь?
2. Вы хотите применить LSTM к временному ряду длиной 10 000 точек с характерным периодом зависимости около 500 шагов. Почему простое увеличение `WINDOW` до 500 может не решить задачу и какие архитектурные альтернативы стоит рассмотреть?
3. Коллега предлагает случайно перемешать ряд перед разбиением на обучение и тест, ссылаясь на то, что так «выборки будут более репрезентативными». Почему это недопустимо для временного ряда и какой конкретный артефакт возникнет в оценке MAE?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это переобучение: модель запомнила обучающие окна, но не обобщила паттерны. Первые шаги: уменьшить `hidden` (размер скрытого слоя) и добавить регуляризацию — `dropout` после LSTM-слоя или `weight_decay` в оптимизаторе.
2. Ячейка памяти LSTM теоретически способна удерживать длинные зависимости, но на практике градиенты через 500 шагов затухают даже у LSTM; кроме того, окно в 500 шагов резко увеличивает вычислительные затраты. Альтернативы: трансформерная архитектура с механизмом внимания, которая явно связывает дальние позиции, или модели пространства состояний (Mamba).
3. При случайном перемешивании обучающие окна будут содержать значения, хронологически стоящие позже тестовых — это прямая утечка будущего в модель. MAE на тесте окажется искусственно заниженной, поскольку модель фактически «знает» будущее в момент обучения; такая оценка не отражает реальную прогностическую способность.
</details>